In [1]:
from google.colab import drive
drive.mount('/content/drive')

In [2]:
!pip install SoccerNet

In [2]:
DATA_DIR = '/content/drive/MyDrive/soccernet'
!mkdir -p {DATA_DIR}

In [3]:
from SoccerNet.Downloader import SoccerNetDownloader

dl = SoccerNetDownloader(LocalDirectory=DATA_DIR)
PASSWORD = "s0cc3rn3t"

In [ ]:
import os
REPO_DIR = '/content/drive/MyDrive/repo'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/atharvjain05/cv-project.git {REPO_DIR}
%cd {REPO_DIR}
!git pull

In [ ]:
# dl.downloadDataTask(task="tracking", split=["train", "test", "challenge"])

In [4]:
# dl.downloadDataTask(task="tracking", split=["train", "test", "challenge"],
                    # password=PASSWORD)

In [5]:
# dl.downloadDataTask(task="calibration", split=["train", "test"])

In [18]:
## Install deps + make ballcond importable
%cd /content/drive/MyDrive/repo
!pip install -q -r requirements-colab.txt
!pip install -q -e .

# Also add src/ to kernel's path so imports work without a restart
import sys
if '/content/drive/MyDrive/repo/src' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/repo/src')

## Explore SoccerNet-Tracking layout

Official clips include **`gameinfo.ini`** next to `gt/` and `seqinfo.ini`. The training loader uses it to:

- pick the **ball** track id (`trackletID_<id>= ball;…`) instead of bbox-area heuristics;
- keep only **field players + goalkeepers** as prediction targets (referees/staff dropped);
- attach **team side** (left vs right in frame) for learned team embeddings.

`gt/gt.txt` is still MOT format (columns 8–10 are `-1`); class names live in **gameinfo**, not in the CSV.

In [19]:
import os
TRACKING_ROOT = '/content/drive/MyDrive/soccernet/tracking'

# List available splits and sequences
for split in ['train', 'test', 'challenge']:
    split_dir = os.path.join(TRACKING_ROOT, split)
    if os.path.isdir(split_dir):
        seqs = sorted(os.listdir(split_dir))
        print(f"{split}: {len(seqs)} sequences — {seqs[:5]}{'...' if len(seqs) > 5 else ''}")
    else:
        print(f"{split}: not found")

In [20]:
import pandas as pd
from pathlib import Path

sample_dir = Path(TRACKING_ROOT) / 'train' / 'SNMOT-060'
sample_gt = sample_dir / 'gt' / 'gt.txt'
sample_gi = sample_dir / 'gameinfo.ini'

df = pd.read_csv(sample_gt, header=None)
df.columns = ['frame', 'id', 'x', 'y', 'w', 'h', 'conf', 'c1', 'c2', 'c3'][: df.shape[1]]
print(f"gt.txt shape: {df.shape}")
print(df.head(5).to_string(index=False))

print(f"\n--- {sample_gi.name} (tracklet lines) ---")
if sample_gi.is_file():
    for line in sample_gi.read_text().splitlines():
        if line.lower().startswith('trackletid_'):
            print(line)
else:
    print("gameinfo.ini missing — use full SoccerNet tracking download.")

## Loader sanity check (`ballcond.data.soccernet`)

Uses **`gameinfo.ini`** for ball id, **field players + GK** only, and **team** labels (`player_team`: 0=left, 1=right, 2=unknown).

In [21]:
from pathlib import Path

cfg = Path('configs')
paths = sorted(cfg.glob('soccernet_exp_*.yaml'))
bc = cfg / 'soccernet_ballcond.yaml'
if bc.is_file():
    paths.append(bc)
for p in paths:
    print(' ', p)


In [22]:
from pathlib import Path
import numpy as np
from ballcond.data.soccernet import load_soccernet_sequence, load_soccernet_split

TRACKING_ROOT = '/content/drive/MyDrive/soccernet/tracking'

seq = load_soccernet_sequence(Path(TRACKING_ROOT) / 'train' / 'SNMOT-060')
print(f"Sequence: {seq.sequence_id}")
print(f"Frames: {seq.num_frames}, Field players + GK columns: {seq.num_players}")
print(f"Ball track id: {seq.meta.get('ball_track_id')} (source: {seq.meta.get('ball_track_source')})")
if seq.has_ball:
    print(f"Ball observed in {seq.ball_mask.sum()}/{seq.num_frames} frames")
if seq.player_team is not None:
    u, c = np.unique(seq.player_team, return_counts=True)
    labels = {0: 'team_left', 1: 'team_right', 2: 'unknown'}
    print("Team label counts:", {labels[int(t)]: int(n) for t, n in zip(u, c)})
print(f"Scale (W, H): {seq.scale}")

In [23]:
train_seqs = load_soccernet_split(Path(TRACKING_ROOT) / 'train')
print(f"Train sequences: {len(train_seqs)}")

n_with_ball = sum(1 for s in train_seqs if s.has_ball)
src_gi = sum(1 for s in train_seqs if s.meta.get('ball_track_source') == 'gameinfo')
print(f"With ball tensor: {n_with_ball}/{len(train_seqs)}  |  ball id from gameinfo: {src_gi}/{len(train_seqs)}")

for s in train_seqs[:5]:
    src = s.meta.get('ball_track_source', '?')
    bid = s.meta.get('ball_track_id')
    vis = f"{s.ball_mask.sum()}/{s.num_frames}" if s.has_ball else "n/a"
    print(f"  {s.sequence_id}: {s.num_frames}f, {s.num_players} targets, ball={bid} ({src}), visible={vis}")

In [24]:
import torch
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))




## Training runs

Configs use **`data.root`** pointing at the tracking folder (same path as `TRACKING_ROOT` above). Each run saves to `results/<run_name>/`.

### Registry: all `model.name` values

| Kind | `model.name` | Notes |
|------|--------------|--------|
| Baseline | `kalman` | Constant-velocity; use `model.name: kalman` (no `kwargs`). |
| Baseline | `lstm` | Per-player LSTM; ignores ball and other agents. |
| Trajectory | `transformer_symmetric` | Factored **time × agent** attention; optional `use_ball`. |
| Trajectory | `transformer_ball_broadcast` | Ball pooled → vector added to all player **time** tokens. |
| Trajectory | `transformer_ballcond` | Players **cross-attend** to encoded ball timeline. |
| Entity | `transformer_entity` | **One token per player** (flatten last `entity_history` steps → linear → self-attn). |
| Entity | `transformer_entity_ball_symmetric` | + **one token per ball track** in the same self-attn set. |
| Entity | `transformer_entity_ball_joint` | + **mean ball embedding** broadcast-added to each player, then self-attn over players only. |

### SoccerNet — time × agent (A–C) + optional BallCond

| Exp | Config | Model |
|-----|--------|-------|
| **A** | `soccernet_exp_ball_symmetric.yaml` | `transformer_symmetric` |
| **B** | `soccernet_exp_ball_broadcast.yaml` | `transformer_ball_broadcast` |
| **C** | `soccernet_exp_no_ball.yaml` | `transformer_symmetric` (`use_ball: false`) |
| Optional | `soccernet_ballcond.yaml` | `transformer_ballcond` |

### SoccerNet — entity-level (D–F)

Match `entity_history` to `data.history` (20 below).

| Exp | Config | Model |
|-----|--------|-------|
| **D** | `soccernet_exp_entity.yaml` | `transformer_entity` |
| **E** | `soccernet_exp_entity_ball_symmetric.yaml` | `transformer_entity_ball_symmetric` |
| **F** | `soccernet_exp_entity_ball_joint.yaml` | `transformer_entity_ball_joint` |

### A — Time × agent, ball as extra agents



In [21]:
!python -m ballcond.train --config configs/soccernet_exp_ball_symmetric.yaml --out results


### B — Ball broadcast embedding


In [22]:
!python -m ballcond.train --config configs/soccernet_exp_ball_broadcast.yaml --out results


### C — No ball input


In [25]:
!python -m ballcond.train --config configs/soccernet_exp_no_ball.yaml --out results


### Optional — BallCond


In [26]:
!python -m ballcond.train --config configs/soccernet_ballcond.yaml --out results


### D — Entity: players only (`transformer_entity`)


In [27]:
!python -m ballcond.train --config configs/soccernet_exp_entity.yaml --out results


### E — Entity: one token per ball track (`transformer_entity_ball_symmetric`)


In [28]:
!python -m ballcond.train --config configs/soccernet_exp_entity_ball_symmetric.yaml --out results


### F — Entity: mean ball vector added to every player token (`transformer_entity_ball_joint`)


In [29]:
!python -m ballcond.train --config configs/soccernet_exp_entity_ball_joint.yaml --out results


## Compare metrics

Loads `metrics.json` from each `results/<run_name>/` after training.


In [30]:
import json
import os

runs = [
    ('exp_A_symmetric', 'soccernet_ball_symmetric_agent'),
    ('exp_B_broadcast', 'soccernet_ball_broadcast'),
    ('exp_C_no_ball', 'soccernet_ball_none'),
    ('optional_ballcond', 'soccernet_ballcond'),
    ('exp_D_entity', 'soccernet_entity_players'),
    ('exp_E_entity_ball_sym', 'soccernet_entity_ball_symmetric'),
    ('exp_F_entity_ball_joint', 'soccernet_entity_ball_joint'),
]
results = {}
for label, dirname in runs:
    path_m = f'results/{dirname}/metrics.json'
    if os.path.exists(path_m):
        with open(path_m) as f:
            results[label] = json.load(f)['val']

if not results:
    print('No metrics yet — run training cells above.')
else:
    keys = sorted(next(iter(results.values())).keys())
    labels = list(results.keys())
    hdr = '{:12}'.format('Metric') + ''.join('{:>16}'.format(lb) for lb in labels)
    print(hdr)
    print('-' * len(str(hdr)))
    for k in keys:
        parts = []
        for lb in labels:
            v = results[lb].get(k, float('nan'))
            parts.append('{:16.5f}'.format(v))
        print('{:12}'.format(k) + ''.join(parts))



## Baselines: Kalman and LSTM

These baselines reuse the same SoccerNet data settings as the transformer experiments, but generate temporary configs in the notebook so we do not need separate YAML files.

- **Kalman:** constant-velocity baseline; no learned parameters and no training epochs.
- **LSTM:** per-player recurrent baseline; ignores ball and inter-agent context.

In [ ]:
from pathlib import Path
from omegaconf import OmegaConf

%cd {REPO_DIR}

BASE_CFG = Path('configs/soccernet_exp_no_ball.yaml')
TMP_CFG_DIR = Path('/tmp/ballcond_baselines')
TMP_CFG_DIR.mkdir(parents=True, exist_ok=True)

def make_soccernet_baseline_config(model_name: str) -> Path:
    cfg = OmegaConf.load(BASE_CFG)
    cfg.data.root = TRACKING_ROOT
    cfg.run_name = f'soccernet_{model_name}'
    cfg.model.name = model_name
    if model_name == 'kalman':
        cfg.model.kwargs = {}
        cfg.train.epochs = 0
        cfg.train.lr = 0.0
        cfg.train.weight_decay = 0.0
    elif model_name == 'lstm':
        cfg.model.kwargs = {
            'hidden_dim': 128,
            'num_layers': 2,
            'dropout': 0.1,
        }
        cfg.train.epochs = 30
        cfg.train.lr = 3.0e-4
        cfg.train.weight_decay = 1.0e-4
    else:
        raise ValueError(model_name)
    out = TMP_CFG_DIR / f'soccernet_{model_name}.yaml'
    OmegaConf.save(cfg, out)
    print(f'Wrote {out}')
    return out

kalman_cfg = make_soccernet_baseline_config('kalman')
lstm_cfg = make_soccernet_baseline_config('lstm')

In [ ]:
%cd {REPO_DIR}
!python -m ballcond.train --config /tmp/ballcond_baselines/soccernet_kalman.yaml --out results

In [ ]:
%cd {REPO_DIR}
!python -m ballcond.train --config /tmp/ballcond_baselines/soccernet_lstm.yaml --out results

## Compare all metrics with baselines

Run this after the baseline cells and any transformer experiment cells you want to include.

In [ ]:
import json
from pathlib import Path

runs = [
    ('kalman', 'soccernet_kalman'),
    ('lstm', 'soccernet_lstm'),
    ('exp_A_symmetric', 'soccernet_ball_symmetric_agent'),
    ('exp_B_broadcast', 'soccernet_ball_broadcast'),
    ('exp_C_no_ball', 'soccernet_ball_none'),
    ('optional_ballcond', 'soccernet_ballcond'),
    ('exp_D_entity', 'soccernet_entity_players'),
    ('exp_E_entity_ball_sym', 'soccernet_entity_ball_symmetric'),
    ('exp_F_entity_ball_joint', 'soccernet_entity_ball_joint'),
]

results = {}
for label, dirname in runs:
    metrics_path = Path('results') / dirname / 'metrics.json'
    if metrics_path.exists():
        with open(metrics_path) as f:
            results[label] = json.load(f)['val']
    else:
        print(f'Missing: {metrics_path}')

if not results:
    print('No metrics yet — run training cells above.')
else:
    keys = sorted(next(iter(results.values())).keys())
    labels = list(results.keys())
    hdr = '{:22}'.format('Metric') + ''.join('{:>24}'.format(lb) for lb in labels)
    print(hdr)
    print('-' * len(hdr))
    for k in keys:
        row = '{:22}'.format(k)
        for lb in labels:
            row += '{:24.5f}'.format(results[lb].get(k, float('nan')))
        print(row)